# Assignment 6: Retrieval-Augmented Generation (RAG) - Machine Learning Knowledge Assistant

## Overview
This notebook implements an end-to-end RAG system that:
1. Loads and processes a Machine Learning book (PDF)
2. Creates embeddings and stores them in a vector database
3. Retrieves relevant content based on queries
4. Generates accurate answers using an LLM

## Assignment Structure
- **Part 1**: Data Understanding & Preprocessing
- **Part 2**: Embedding & Vector Database
- **Part 3**: Retrieval Pipeline
- **Part 4**: Answer Generation (RAG)
- **Part 5**: End-to-End RAG Application

---
## Part 1: Data Understanding & Preprocessing

### 1.1 Install Required Libraries
First, let's install all necessary packages.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install langchain langchain-community langchain-openai langchain-text-splitters pypdf faiss-cpu openai python-dotenv

### 1.2 Import Libraries

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os, re, json, time, warnings
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from langchain_openai import OpenAIEmbeddings, OpenAI
import getpass
from langchain_community.vectorstores import FAISS
from langchain.schema import Document
from langchain_core.prompts import PromptTemplate

warnings.filterwarnings('ignore')
from pypdf import PdfReader

print("✓ All libraries imported successfully!")

### 1.3 Define Helper Functions for PDF Processing

In [ ]:
def load_pdf(path: str) -> Tuple[List[str], Dict]:
    """Load PDF and extract text from all pages with metadata."""
    reader = PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        pages.append(text)

    meta = {
        "num_pages": len(pages),
        "total_chars": sum(len(p) for p in pages),
        "avg_chars_per_page": int(np.mean([len(p) for p in pages])),
        "empty_pages": sum(1 for p in pages if len(p.strip()) < 50),
        "title": reader.metadata.get("/Title", "Unknown"),
        "author": reader.metadata.get("/Author", "Unknown"),
    }
    return pages, meta


def analyze_text_quality(pages: List[str]) -> Tuple[Dict, List[str]]:
    """Analyze text quality issues in the PDF pages."""
    issues = {
        "hyphenated_words": 0,
        "repeated_spaces": 0,
        "page_numbers_only": 0,
        "header_footer_noise": 0,
    }
    samples = []
    for p in pages:
        issues["hyphenated_words"] += len(re.findall(r'\w+-\n\w+', p))
        issues["repeated_spaces"] += len(re.findall(r' {3,}', p))
        issues["page_numbers_only"] += 1 if re.fullmatch(r'\s*\d+\s*', p) else 0
        if p.strip():
            samples.append(p[:120].replace('\n', ' '))
    return issues, samples[:3]


def clean_text(text: str) -> str:
    """Clean text by removing hyphenation, extra spaces, and page numbers."""
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)  # Remove hyphenation
    text = re.sub(r' {2,}', ' ', text)  # Normalize spaces
    text = re.sub(r'\n{3,}', '\n\n', text)  # Normalize line breaks
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.M)  # Remove page numbers
    return text.strip()

print("✓ Helper functions defined successfully!")

### 1.4 Load PDF Document and Extract Metadata

In [ ]:
# Load the PDF document
pages, doc_meta = load_pdf('./intro-to-ml.pdf')

print(f"📖 PDF Document Loaded Successfully!")
print(f"{'='*60}")
print(f"Title          : {doc_meta['title']}")
print(f"Author         : {doc_meta['author']}")
print(f"Number of Pages: {doc_meta['num_pages']}")
print(f"Total Characters: {doc_meta['total_chars']:,}")
print(f"Avg Chars/Page : {doc_meta['avg_chars_per_page']:,}")
print(f"Empty Pages    : {doc_meta['empty_pages']}")
print(f"{'='*60}")

### 1.5 Analyze Text Quality Issues

In [ ]:
quality_issues, sample_texts = analyze_text_quality(pages)

print("🔍 Text Quality Analysis:")
print(f"{'='*60}")
for issue, count in quality_issues.items():
    print(f"{issue:<28}: {count}")
print(f"{'='*60}")

print("\n📄 Sample Text Excerpts (first 3 pages):")
for i, sample in enumerate(sample_texts, 1):
    print(f"\nPage {i}: {sample}...")

### 1.6 Clean Text Content

In [ ]:
# Clean all pages
cleaned_pages = [clean_text(p) for p in pages]

# Combine into full text, filtering out very short pages
full_text = "\n\n".join(p for p in cleaned_pages if len(p) > 50)

print(f"✓ Text cleaned successfully!")
print(f"Total cleaned text length: {len(full_text):,} characters")
print(f"\n📝 Sample of cleaned text (first 500 chars):")
print(f"{'-'*60}")
print(full_text[:500])
print(f"{'-'*60}")

### 1.7 Split Text into Chunks

We'll use `RecursiveCharacterTextSplitter` to split the text into manageable chunks.
- **Chunk Size**: 800 characters (approximately 150-200 tokens)
- **Chunk Overlap**: 100 characters to maintain context continuity

In [ ]:
# Configure text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Split text into chunks
chunks = text_splitter.split_text(full_text)

print(f"✓ Text split into chunks successfully!")
print(f"{'='*60}")
print(f"Number of chunks created: {len(chunks)}")
print(f"Average chunk size: {int(np.mean([len(c) for c in chunks]))} characters")
print(f"Min chunk size: {min(len(c) for c in chunks)} characters")
print(f"Max chunk size: {max(len(c) for c in chunks)} characters")
print(f"{'='*60}")

print(f"\n📦 Sample Chunks:")
for i, chunk in enumerate(chunks[:3], 1):
    print(f"\n--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk[:200] + "...")

---
## Part 2: Embedding & Vector Database

### 2.1 Set Up OpenAI API Key

In [ ]:
# Set up OpenAI API key
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")
    
print("✓ OpenAI API key configured!")

### 2.2 Initialize Embedding Model

We'll use **OpenAI's text-embedding-3-small** model:
- **Dimensions**: 1536
- **Cost-effective** for large documents
- **High quality** semantic representations

In [ ]:
# Initialize embedding model
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.environ.get("OPENAI_API_KEY")
)

# Test the embedding model
test_embedding = embeddings.embed_query("What is machine learning?")

print(f"✓ Embedding model initialized successfully!")
print(f"{'='*60}")
print(f"Model: text-embedding-3-small")
print(f"Embedding dimensions: {len(test_embedding)}")
print(f"Sample embedding (first 10 values): {test_embedding[:10]}")
print(f"{'='*60}")

### 2.3 Create Document Objects for Vector Store

In [ ]:
# Convert text chunks to Document objects with metadata
documents = [
    Document(
        page_content=chunk,
        metadata={"chunk_id": i, "source": "intro-to-ml.pdf"}
    )
    for i, chunk in enumerate(chunks)
]

print(f"✓ Created {len(documents)} Document objects")
print(f"\n📄 Sample Document:")
print(f"Content: {documents[0].page_content[:150]}...")
print(f"Metadata: {documents[0].metadata}")

### 2.4 Generate Embeddings and Create FAISS Vector Database

**FAISS** (Facebook AI Similarity Search) is a library for efficient similarity search:
- Fast vector search even with millions of vectors
- Memory efficient
- Supports various distance metrics

In [ ]:
print("🔄 Generating embeddings and creating vector database...")
print("This may take a few minutes depending on the document size...")

start_time = time.time()

# Create FAISS vector store from documents
vectorstore = FAISS.from_documents(documents, embeddings)

elapsed_time = time.time() - start_time

print(f"\n✓ Vector database created successfully!")
print(f"{'='*60}")
print(f"Number of vectors: {len(documents)}")
print(f"Embedding dimensions: {len(test_embedding)}")
print(f"Time taken: {elapsed_time:.2f} seconds")
print(f"Vector store type: FAISS")
print(f"{'='*60}")

### 2.5 Test Similarity Search

Let's test the vector database with a sample query.

In [ ]:
# Test query
test_query = "What is supervised learning?"

# Perform similarity search
similar_docs = vectorstore.similarity_search_with_score(test_query, k=3)

print(f"🔍 Test Query: '{test_query}'")
print(f"{'='*60}")

for i, (doc, score) in enumerate(similar_docs, 1):
    print(f"\n📄 Result {i} (Similarity Score: {score:.4f}):")
    print(f"Chunk ID: {doc.metadata['chunk_id']}")
    print(f"Content Preview: {doc.page_content[:200]}...")
    print(f"{'-'*60}")

### 2.6 Save Vector Database (Optional)

Save the vector database to disk for future use.

In [ ]:
# Save vector database
vectorstore.save_local("faiss_ml_book_index")

print("✓ Vector database saved to 'faiss_ml_book_index'")

# To load later:
# vectorstore = FAISS.load_local("faiss_ml_book_index", embeddings)

---
## Part 3: Retrieval Pipeline

### 3.1 Implement Retrieval Function

In [ ]:
def retrieve_relevant_chunks(query: str, k: int = 5) -> List[Tuple[Document, float]]:
    """
    Retrieve top-k most relevant chunks for a given query.
    
    Args:
        query: User's question
        k: Number of chunks to retrieve
        
    Returns:
        List of (Document, similarity_score) tuples
    """
    results = vectorstore.similarity_search_with_score(query, k=k)
    return results

print("✓ Retrieval function defined successfully!")

### 3.2 Test Retrieval with Sample Queries

In [ ]:
# Test with different queries
test_queries = [
    "What is the difference between supervised and unsupervised learning?",
    "Explain neural networks",
    "What is gradient descent?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"🔍 Query: {query}")
    print(f"{'='*60}")
    
    results = retrieve_relevant_chunks(query, k=3)
    
    for i, (doc, score) in enumerate(results, 1):
        print(f"\n📄 Chunk {i} (Score: {score:.4f}):")
        print(f"   {doc.page_content[:150]}...")
        print()

### 3.3 Experiment with Different K Values

Compare retrieval quality with k=3, 5, and 7.

In [ ]:
# Experiment with different k values
query = "What are convolutional neural networks?"
k_values = [3, 5, 7]

print(f"📊 Comparing different K values for query: '{query}'")
print(f"{'='*80}\n")

for k in k_values:
    results = retrieve_relevant_chunks(query, k=k)
    print(f"K = {k}:")
    print(f"  Retrieved {len(results)} chunks")
    print(f"  Score range: {results[-1][1]:.4f} to {results[0][1]:.4f}")
    print(f"  Average score: {np.mean([score for _, score in results]):.4f}")
    print()

### 3.4 Evaluate Retrieval Quality

Display full content of retrieved chunks to assess relevance.

In [ ]:
# Detailed retrieval evaluation
evaluation_query = "What is overfitting and how to prevent it?"

print(f"🔬 Detailed Retrieval Evaluation")
print(f"Query: {evaluation_query}")
print(f"{'='*80}\n")

results = retrieve_relevant_chunks(evaluation_query, k=5)

for i, (doc, score) in enumerate(results, 1):
    print(f"📄 Retrieved Chunk {i}")
    print(f"   Similarity Score: {score:.4f}")
    print(f"   Chunk ID: {doc.metadata['chunk_id']}")
    print(f"   Content Length: {len(doc.page_content)} characters")
    print(f"\n   Full Content:")
    print(f"   {'-'*76}")
    print(f"   {doc.page_content}")
    print(f"   {'-'*76}\n")

---
## Part 4: Answer Generation (RAG)

### 4.1 Design RAG Prompt Template

The prompt is designed to:
- Instruct the LLM to answer based **only on the provided context**
- Minimize hallucination
- Encourage citing specific information from the context

In [ ]:
# Create RAG prompt template
rag_template = """You are an expert Machine Learning assistant. Answer the question based ONLY on the following context from a Machine Learning textbook.

Context:
{context}

Question: {question}

Instructions:
1. Answer the question using ONLY information from the context provided above
2. If the context doesn't contain enough information to answer the question, say "I don't have enough information in the provided context to answer this question."
3. Be specific and cite relevant details from the context
4. Do not make up or infer information that is not explicitly stated in the context
5. Keep your answer concise but comprehensive

Answer:"""

rag_prompt = PromptTemplate(
    template=rag_template,
    input_variables=["context", "question"]
)

print("✓ RAG prompt template created successfully!")
print(f"\n📋 Prompt Template Preview:")
print(f"{'-'*60}")
print(rag_template[:300] + "...")
print(f"{'-'*60}")

### 4.2 Initialize Language Model

In [ ]:
# Initialize OpenAI LLM
llm = OpenAI(
    model="gpt-3.5-turbo-instruct",
    temperature=0.2,  # Low temperature for more factual, grounded responses
    max_tokens=500,
    openai_api_key=os.environ.get("OPENAI_API_KEY")
)

print("✓ Language Model initialized successfully!")
print(f"{'='*60}")
print(f"Model: gpt-3.5-turbo-instruct")
print(f"Temperature: 0.2 (low for factual responses)")
print(f"Max Tokens: 500")
print(f"{'='*60}")

### 4.3 Implement RAG Answer Generation Function

In [ ]:
def generate_answer(question: str, k: int = 5) -> Dict[str, any]:
    """
    Generate an answer using RAG approach.
    
    Args:
        question: User's question
        k: Number of chunks to retrieve
        
    Returns:
        Dictionary containing question, answer, context, and sources
    """
    # Step 1: Retrieve relevant chunks
    retrieved_docs = retrieve_relevant_chunks(question, k=k)
    
    # Step 2: Combine chunks into context
    context = "\n\n".join([doc.page_content for doc, _ in retrieved_docs])
    
    # Step 3: Format prompt
    prompt = rag_prompt.format(context=context, question=question)
    
    # Step 4: Generate answer
    answer = llm(prompt)
    
    # Step 5: Prepare response
    response = {
        "question": question,
        "answer": answer.strip(),
        "context": context,
        "sources": [
            {
                "chunk_id": doc.metadata['chunk_id'],
                "similarity_score": float(score),
                "content": doc.page_content
            }
            for doc, score in retrieved_docs
        ],
        "num_sources": len(retrieved_docs)
    }
    
    return response

print("✓ RAG answer generation function defined successfully!")

### 4.4 Generate Sample Answers

Test the RAG system with various questions.

In [ ]:
# Test question
test_question = "What is supervised learning and how does it work?"

print(f"🤖 Generating answer using RAG...")
print(f"{'='*80}\n")

response = generate_answer(test_question, k=5)

print(f"❓ Question: {response['question']}")
print(f"\n✅ Answer:")
print(f"{'-'*80}")
print(response['answer'])
print(f"{'-'*80}")

print(f"\n📚 Retrieved {response['num_sources']} source chunks")
print(f"\n🔗 Top 3 Source Chunks:")
for i, source in enumerate(response['sources'][:3], 1):
    print(f"\n   Source {i} (Chunk ID: {source['chunk_id']}, Score: {source['similarity_score']:.4f}):")
    print(f"   {source['content'][:200]}...")
print()

### 4.5 Validate Answer Grounding

Test with multiple questions to ensure answers are grounded in context.

In [ ]:
# Test multiple questions to validate grounding
validation_questions = [
    "What is the difference between classification and regression?",
    "Explain the concept of overfitting",
    "What are the main types of neural network layers?"
]

print(f"🧪 Validating Answer Grounding with Multiple Questions")
print(f"{'='*80}\n")

for i, question in enumerate(validation_questions, 1):
    print(f"\n{'='*80}")
    print(f"Test {i}: {question}")
    print(f"{'='*80}")
    
    response = generate_answer(question, k=5)
    
    print(f"\n💬 Answer:")
    print(response['answer'])
    print(f"\n📊 Number of sources used: {response['num_sources']}")
    print()

---
## Part 5: End-to-End RAG Application

### 5.1 Create Complete RAG Pipeline Class

In [ ]:
class MLKnowledgeAssistant:
    """
    Complete RAG-based Machine Learning Knowledge Assistant.
    """
    
    def __init__(self, vectorstore, embeddings, llm, rag_prompt):
        self.vectorstore = vectorstore
        self.embeddings = embeddings
        self.llm = llm
        self.rag_prompt = rag_prompt
        self.conversation_history = []
    
    def ask(self, question: str, k: int = 5, show_sources: bool = True) -> str:
        """
        Ask a question and get an answer.
        
        Args:
            question: User's question
            k: Number of source chunks to retrieve
            show_sources: Whether to display source information
            
        Returns:
            Generated answer
        """
        # Retrieve relevant chunks
        retrieved_docs = self.vectorstore.similarity_search_with_score(question, k=k)
        
        # Combine into context
        context = "\n\n".join([doc.page_content for doc, _ in retrieved_docs])
        
        # Generate answer
        prompt = self.rag_prompt.format(context=context, question=question)
        answer = self.llm(prompt).strip()
        
        # Store in history
        self.conversation_history.append({
            "question": question,
            "answer": answer,
            "sources": [(doc.metadata['chunk_id'], score) for doc, score in retrieved_docs]
        })
        
        # Display results
        print(f"\n{'='*80}")
        print(f"❓ Question: {question}")
        print(f"{'='*80}")
        print(f"\n✅ Answer:")
        print(answer)
        
        if show_sources:
            print(f"\n📚 Sources Used: {len(retrieved_docs)} chunks")
            for i, (doc, score) in enumerate(retrieved_docs[:3], 1):
                print(f"   {i}. Chunk {doc.metadata['chunk_id']} (Similarity: {score:.4f})")
        
        print(f"\n{'='*80}\n")
        
        return answer
    
    def get_history(self):
        """Get conversation history."""
        return self.conversation_history
    
    def clear_history(self):
        """Clear conversation history."""
        self.conversation_history = []
        print("✓ Conversation history cleared!")

# Initialize the assistant
ml_assistant = MLKnowledgeAssistant(vectorstore, embeddings, llm, rag_prompt)

print("✓ ML Knowledge Assistant initialized successfully!")
print("You can now ask questions using: ml_assistant.ask('your question')")

### 5.2 Interactive Q&A Session - Conceptual Questions

In [ ]:
# Conceptual Questions
print("🧠 Testing with Conceptual Questions")
print("="*80 + "\n")

ml_assistant.ask("What is machine learning?")
ml_assistant.ask("What is the difference between supervised and unsupervised learning?")

### 5.3 Interactive Q&A Session - Technical Details

In [ ]:
# Technical Details
print("\n🔧 Testing with Technical Questions")
print("="*80 + "\n")

ml_assistant.ask("How does gradient descent work?")
ml_assistant.ask("What are the key components of a neural network?")

### 5.4 Interactive Q&A Session - Algorithm Explanations

In [ ]:
# Algorithm Explanations
print("\n📐 Testing with Algorithm Questions")
print("="*80 + "\n")

ml_assistant.ask("Explain decision trees")
ml_assistant.ask("What is k-means clustering?")

### 5.5 Interactive Q&A Session - Comparison Questions

In [ ]:
# Comparison Questions
print("\n⚖️ Testing with Comparison Questions")
print("="*80 + "\n")

ml_assistant.ask("What is the difference between bagging and boosting?")
ml_assistant.ask("Compare CNNs and RNNs")

### 5.6 View Conversation History

In [ ]:
# View conversation history
history = ml_assistant.get_history()

print(f"📜 Conversation History")
print(f"{'='*80}")
print(f"Total questions asked: {len(history)}\n")

for i, entry in enumerate(history, 1):
    print(f"{i}. Q: {entry['question']}")
    print(f"   Sources used: {len(entry['sources'])} chunks")
    print()

### 5.7 System Performance Demonstration

Let's demonstrate a complete end-to-end query with full details.

In [ ]:
# Detailed demonstration
demo_question = "What is overfitting and how can we prevent it?"

print(f"🎯 Complete RAG Pipeline Demonstration")
print(f"{'='*80}\n")

# Generate detailed response
response = generate_answer(demo_question, k=5)

print(f"❓ Question: {response['question']}")
print(f"\n✅ Generated Answer:")
print(f"{'-'*80}")
print(response['answer'])
print(f"{'-'*80}")

print(f"\n📊 Retrieval Statistics:")
print(f"   - Number of chunks retrieved: {response['num_sources']}")
print(f"   - Context length: {len(response['context'])} characters")

print(f"\n📚 Retrieved Context Chunks:")
for i, source in enumerate(response['sources'], 1):
    print(f"\n   Chunk {i} (ID: {source['chunk_id']}, Score: {source['similarity_score']:.4f}):")
    print(f"   {'-'*76}")
    print(f"   {source['content']}")
    print(f"   {'-'*76}")

### 5.8 Custom Query Interface

Use this cell to ask your own questions!

In [ ]:
# Ask your own question here!
# Change the question variable to ask different questions

your_question = "What is deep learning?"

ml_assistant.ask(your_question, k=5, show_sources=True)

---
## Summary & Conclusion

### ✅ Assignment Completion Checklist

#### Part 1: Data Understanding & Preprocessing
- ✅ Loaded PDF document and extracted metadata
- ✅ Analyzed document structure (pages, characters, quality issues)
- ✅ Cleaned text content (hyphenation, spaces, page numbers)
- ✅ Split text into chunks (800 chars, 100 overlap)

#### Part 2: Embedding & Vector Database
- ✅ Generated embeddings using OpenAI text-embedding-3-small
- ✅ Created FAISS vector database
- ✅ Implemented similarity search mechanism
- ✅ Saved vector database for reuse

#### Part 3: Retrieval Pipeline
- ✅ Implemented query embedding conversion
- ✅ Built retrieval function with configurable k
- ✅ Experimented with different k values (3, 5, 7)
- ✅ Evaluated retrieval quality with similarity scores

#### Part 4: Answer Generation (RAG)
- ✅ Designed RAG prompt template to minimize hallucination
- ✅ Initialized OpenAI LLM with low temperature
- ✅ Implemented context-based answer generation
- ✅ Validated answer grounding with multiple queries

#### Part 5: End-to-End RAG Application
- ✅ Built complete MLKnowledgeAssistant class
- ✅ Created interactive Q&A interface
- ✅ Demonstrated with diverse query types:
  - Conceptual questions
  - Technical details
  - Algorithm explanations
  - Comparison questions
- ✅ Tracked conversation history

### 🎓 Key Learnings

1. **Document Processing**: Proper text cleaning significantly improves retrieval quality
2. **Chunking Strategy**: Chunk size and overlap balance context completeness with specificity
3. **Vector Search**: FAISS provides fast similarity search even with many documents
4. **Prompt Engineering**: Well-designed prompts minimize hallucination and improve grounding
5. **RAG Benefits**: Combining retrieval with generation provides accurate, contextual answers

### 🚀 Potential Enhancements

- Add re-ranking for better retrieval results
- Implement hybrid search (keyword + semantic)
- Add conversation memory for follow-up questions
- Include confidence scores in answers
- Build a web interface with Streamlit or Gradio
- Add support for multiple PDFs
- Implement query expansion for better retrieval

### 📊 System Performance

- **Retrieval Speed**: Fast similarity search with FAISS
- **Answer Quality**: High accuracy due to grounded responses
- **Flexibility**: Configurable k-value for different use cases
- **Scalability**: Can handle large documents efficiently

In [ ]:
print("🎉 Assignment 6 Complete!")
print("="*80)
print("You have successfully built an end-to-end RAG application!")
print("The ML Knowledge Assistant is ready to answer your questions.")
print("="*80)